In [ ]:
%load_ext autoreload
%autoreload 2
import waffles
import numpy as np
import json
import shutil 
from tqdm import tqdm
from pathlib import Path

from waffles.input_output.hdf5_structured import load_structured_waveformset
from waffles.data_classes.Waveform import Waveform
from waffles.data_classes.WaveformSet import WaveformSet
from waffles.data_classes.BasicWfAna import BasicWfAna
from waffles.data_classes.IPDict import IPDict
from waffles.data_classes.UniqueChannel import UniqueChannel
from waffles.data_classes.ChannelWsGrid import ChannelWsGrid
from waffles.utils.baseline.baseline import SBaseline
from waffles.np02_utils.AutoMap import generate_ChannelMap, dict_uniqch_to_module, dict_module_to_uniqch, getModuleName
from waffles.np02_utils.PlotUtils import np02_gen_grids, plot_grid, plot_detectors, genhist, fithist, runBasicWfAnaNP02
from waffles.np02_utils.PlotUtils import plot_average_spe, matplotlib_plot_WaveformSetGrid
from waffles.np02_utils.AutoMap import expand_modules

from run_calibration import open_processed, filter_bad_waveforms, parse_runslist, detector_groups
from run_calibration import DEFAULT_DATADIR, ENDPOINT_BY_DETTYPE, SPE_PRETRIGGER, SPE_POSTTRIGGER, SPE_THRESHOLD, SPE_MAXIMUM_BEGIN, SPE_MAXIMUM_END
from run_calibration import plot_average_spe_and_store, get_output_folder, plotly_lines_to_png, save_results

In [ ]:
dettype = "membrane"
dettype = "cathode"

runslist = "runslist.txt"
# Modules to analyse (only the ones of dettype are used). "C"/"M" = all cathode/membrane modules
detectors = ["C"]
# or use this
# detectors = ["C1", "C2(1)"]




## Only change if necessary
datadir = DEFAULT_DATADIR
det = "VD_Cathode_PDS" if dettype == "cathode" else "VD_Membrane_PDS"
endpoint = ENDPOINT_BY_DETTYPE[dettype]

# Way to low... keep scrollng
groupLow, groupHig = detector_groups(dettype)
groupall = groupLow+groupHig
try:
    tmpep = list(wfset_full.get_set_of_endpoints())
    if endpoint != tmpep[0]: # recreate the wfset, only one endpoint at a time
        del wfset_full
        wfset_full = None
except:
    wfset_full=None

In [ ]:
_, run_to_module = parse_runslist(Path(runslist))
selected = expand_modules(detectors, list(dict_module_to_uniqch.keys()))
run_to_module = { r: [ m for m in modules if m in selected ] for r, modules in run_to_module[dettype].items() }
run_to_module = { r: modules for r, modules in run_to_module.items() if modules }
print(run_to_module)

run_to_unich = { r: [ dict_module_to_uniqch[m].channel for m in modules ] for r, modules in run_to_module.items() }
channels = [ x for v in run_to_unich.values() for x in v]

In [ ]:
import copy
import time
def select_channels(waveform: Waveform, channels: list) -> bool:
    if waveform.channel not in channels:
        return False
    if np.any(waveform.adcs[0:500] > 16000):
        return False
    if np.any(waveform.adcs[0:500] < 100 ):
        return False
    return True
def remove_channel_when_run_change(waveform: Waveform, channels_to_remove = []) -> bool:
    if waveform.channel in channels_to_remove:
        return False
    return True
    
def create_wfset(run_to_unich, endpoint, wfset_alread_there=None):

    nwaveforms = None
    wfset_full:WaveformSet = wfset_alread_there 

    if wfset_full is not None:
        for run in wfset_full.runs:
            if run not in run_to_unich.keys():
                if len(wfset_full.runs) != 1:
                    print(f"Removing unused run {run}")
                    wfset_full = WaveformSet.from_filtered_WaveformSet(wfset_full, remove_channel_when_run_change, wfset_full.available_channels[run][endpoint], show_progress=True)
                else:
                    wfset_full = None

    for run, channels in run_to_unich.items():
        if wfset_full is not None:
            redo = True
            
            if run in wfset_full.runs:
                if list(wfset_full.available_channels[run][endpoint]) == sorted(channels):
                    print(f"Run {run} already there :)")
                    redo=False
                
            if redo: 
                try:
                    wfset_full = WaveformSet.from_filtered_WaveformSet(wfset_full, remove_channel_when_run_change, channels_to_remove=channels, show_progress=True)
                except:
                    wfset_full = None
            else:
                continue
                
        wfset = open_processed(run, dettype, datadir, channels, [endpoint], nwaveforms=nwaveforms, verbose=True, mergefiles=False)
        # wfset = WaveformSet.from_filtered_WaveformSet(wfset, select_channels, channels)
        if wfset_full is None:
            wfset_full = copy.deepcopy(wfset)
        else:
            wfset_full.merge(copy.deepcopy(wfset))
        print(f"Loaded run {run}")
    return wfset_full
    
start = time.time()
wfset_full = create_wfset(run_to_unich, endpoint, wfset_full)
end = time.time()
print(end - start)

In [ ]:
print(len(wfset_full.waveforms))
wfset = WaveformSet.from_filtered_WaveformSet(wfset_full, filter_bad_waveforms, show_progress=True)
print(len(wfset.waveforms))

In [ ]:
from pathlib import Path
from datetime import datetime 

histargs = dict(    
    # doprocess = False,
    dofit = True,
    # show_progress = True,
    fitmultigauss = True,
    # verbosemultigauss = True,
    configyaml = "ch_snr_parameters.yaml",
)
setupargs=dict(
# default 1300x1300
    width=1300,
    height=650,
    showplots=True
)
htmlname = None

# timenow = datetime.now().strftime('%Y%m%d_%H%M%S') 
# htmlname = Path(f"./calibration_{dettype}_{timenow}.html")

# # Any detector can be processed alone if all were processed already
# detector=groupLow
# plot_detectors(wfset_full, detector, plot_function=fithist, html=None, wf_func=histargs)
  
# detector=["C6(1)", "C6(2)"]
# plot_detectors(wfset, detector, plot_function=fithist, html=None, wf_func=histargs)

detector=groupLow+groupHig
figs = plot_detectors(wfset, detector, plot_function=fithist, html=htmlname, wf_func=histargs, return_fig=True, **setupargs)
for fig_hist, *_ in figs:
    fig_hist.show()
# detector=groupHig
# plot_detectors(wfset, detector, plot_function=fithist, html=htmlname, wf_func=histargs, **setupargs)

In [ ]:
argsheat = dict(
    mode="heatmap",
    analysis_label="std",
    adc_range_above_baseline=500,
    adc_range_below_baseline=-150,
    adc_bins=125,
    time_bins=wfset.points_per_wf//2,
    filtering=4,
    share_y_scale=False,
    share_x_scale=True,
    wfs_per_axes=5000,
    zlog=True,
    width=1300,
    heigth=650
)
detector=groupLow
plot_detectors(wfset, detector, **argsheat)

detector=groupHig
plot_detectors(wfset, detector, **argsheat)

In [ ]:
spe_amplitudes = {}
func_params = dict(
    spe_amplitudes=spe_amplitudes,
    pretrigger=SPE_PRETRIGGER,
    posttrigger=SPE_POSTTRIGGER,
    threshold=SPE_THRESHOLD,
    maximum_begin=SPE_MAXIMUM_BEGIN,
    maximum_end=SPE_MAXIMUM_END,
)
detector = groupLow+groupHig
fig_spe, _ = matplotlib_plot_WaveformSetGrid(wfset, plot_function=plot_average_spe_and_store, detector=detector, func_params=func_params)
# detector = groupHig
# matplotlib_plot_WaveformSetGrid(wfset_full, plot_function=plot_average_spe, detector=detector, func_params=func_params)

In [ ]:
# Save the results in the same way as run_calibration.py (results/YYYYMMDD_N/)
outdir = "results"
date, _ = parse_runslist(Path(runslist))
folder = get_output_folder(Path(outdir), date, Path(runslist))

# for fig_hist, rows, cols, *_ in figs:
#     plotly_lines_to_png(fig_hist, rows, cols, folder / f"charge_hist_{dettype}.png")
# fig_spe.savefig(folder / f"average_spe_{dettype}.png")
# print(f"Saved {folder / f'average_spe_{dettype}.png'}")
save_results(wfset, groupall, spe_amplitudes, dettype, folder)

In [ ]:
wfset_ch = ChannelWsGrid.clusterize_waveform_set(wfset_full)

In [ ]:
wfs = wfset_ch[endpoint][32]

In [ ]:
list_of_bad_apples = []
for i, w in enumerate(wfs.waveforms):
    v = w.adcs - w.analyses['std'].result['baseline']
    # if abs(w.analyses['std'].result['integral']) < 200:
    list_of_bad_apples.append(v)
        # print(i)

In [ ]:
import matplotlib.pyplot as plt
a+=1
plt.plot(list_of_bad_apples[a])